In [1]:
# 1. Install Ollama (zstd is required by the installer but missing from the base Colab image)
!apt-get -qq update && apt-get -qq install -y zstd
!apt-get update && apt-get install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.lau

In [2]:
import os, subprocess, time

env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0"  # <--- ADD THIS LINE

subprocess.run(["bash", "-c", "fuser -k 11434/tcp || true"])
time.sleep(3)

ollama_process = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)
print("Ollama server starting (pid:", ollama_process.pid, ")")

status = subprocess.run(
    ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://localhost:11434/api/tags"],
    capture_output=True, text=True,
).stdout
print("Local check (expect 200):", status)

Ollama server starting (pid: 3425 )
Local check (expect 200): 200


In [3]:
!ollama pull minicpm-v

In [4]:
# 3. Pull the models used by CanvasUI (src/lib/models.ts)
# !ollama pull qwen2.5vl:7b


In [5]:
# !ollama pull qwen2.5-coder:14b

In [6]:
#!ollama pull qwen3.6:27b

In [7]:
# !ollama pull moondream

In [ ]:
!pip install -q fastapi uvicorn pycloudflared

from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
import uvicorn, uuid, threading, base64, requests, json

app = FastAPI()
jobs = {}

class SubmitPayload(BaseModel):
    image_b64: str
    model: str = "minicpm-v"
    prompt: str = "Extract all text from this resume image."

def _run_inference(job_id: str, payload: SubmitPayload):
    try:
        response = requests.post("http://localhost:11434/api/chat", json={
            "model": payload.model,
            "messages": [{"role": "user", "content": payload.prompt, "images": [payload.image_b64]}],
            "stream": True,
        }, stream=True, timeout=600)
        content = ""
        for line in response.iter_lines():
            if line:
                chunk = json.loads(line)
                content += chunk.get("message", {}).get("content", "")
        jobs[job_id] = {"status": "done", "result": content.strip()}
    except Exception as e:
        jobs[job_id] = {"status": "error", "result": str(e)}

@app.post("/submit")
async def submit(payload: SubmitPayload, background_tasks: BackgroundTasks):
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "pending", "result": None}
    background_tasks.add_task(_run_inference, job_id, payload)
    return {"job_id": job_id}

@app.get("/result/{job_id}")
async def get_result(job_id: str):
    return jobs.get(job_id, {"status": "not_found"})

# Start FastAPI in background thread
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True
).start()

import time; time.sleep(2)  # let server start

# Tunnel points to port 8000 (FastAPI), NOT 11434 (Ollama)
from pycloudflared import try_cloudflare
tunnel_url = try_cloudflare(port=8000)
print("Paste this URL in the frontend as your Ollama URL:")
print(tunnel_url.tunnel)


In [ ]:
# # 4. Expose port 11434 publicly via Cloudflare Tunnel
# !pip install -q pycloudflared

# from pycloudflared import try_cloudflare

# # Cloudflare doesn't require an authentication token for quick/anonymous tunnels!
# # Start the tunnel targeting the Ollama port
# tunnel_url = try_cloudflare(port=11434)

# print("Your public Ollama API endpoint is:")
# print(tunnel_url)


Download cloudflared...:   0%|          | 0/39244939 [00:00<?, ?it/s]

 * Running on https://journal-weight-remainder-discounts.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics
Your public Ollama API endpoint is:
Urls(tunnel='https://journal-weight-remainder-discounts.trycloudflare.com', metrics='http://127.0.0.1:20241/metrics', process=<Popen: returncode: None args: ['/usr/local/lib/python3.12/dist-packages/pyc...>)


In [9]:
!ollama list


NAME                ID              SIZE      MODIFIED       
minicpm-v:latest    c92bfad01205    5.5 GB    12 seconds ago    


In [10]:
print("abc")

abc


In [11]:
!ollama list


NAME                ID              SIZE      MODIFIED       
minicpm-v:latest    c92bfad01205    5.5 GB    12 seconds ago    


In [12]:
!ollama list

NAME                ID              SIZE      MODIFIED       
minicpm-v:latest    c92bfad01205    5.5 GB    13 seconds ago    


In [13]:
print("abc")

abc
